In [ ]:
import os
import pickle
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import subprocess

from utils import compose_windows, make_names_dict

from sklearn.base import clone
from sklearn.model_selection import KFold, StratifiedKFold
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_curve, auc, accuracy_score

## Dobra wersja wykorzystująca `statsmodels` (20.01.26)

## TODO

Zrobić to w statsmodels, tak jak opisano na Mattermost. W statsmodels trzeba ręczenie dodać wyraz wolny w modelu liniowym.

Dodać do df informacje o wagach i zapisać to w postaci long

Narysować współczynninki z regresji logistycznej z przedziałami ufności

Sprawdzić czemu w obecnych wynikach wszystkie wiersze wychodzą identycznie

### Logistic Regression regularization trajecotries:

Here we only investigate what happens to the model weights as we turn up the L2. L1 and ElasticNet are ignored only for simplicity and no other reason.

For each tissue, a DataFrame is created that contains:
1. Metrics (mean and std of ACC and AUC ROC) computed on **CROSS-VALIDATION**
2. Lists of most important motifs based on weights of the model **trained on all of the data**

## Zła wersja wykorzystująca `sklearn` (13.01.26)

### Logistic Regression regularization trajecotries:

Here we only investigate what happens to the model weights as we turn up the L2. L1 and ElasticNet are ignored only for simplicity and no other reason.

For each tissue, a DataFrame is created that contains:
1. Metrics (mean and std of ACC and AUC ROC) computed on **CROSS-VALIDATION**
2. Lists of most important motifs based on weights of the model **trained on all of the data**

In [ ]:
windows=["06-08", "10-12", "14-16"]
tissues=["Neuroblasts", "Neurons", "Glia"]
model_name = "Logistic Regression"
model_short = "LR"

n_splits = 10   # cross-validation splits

c_values = np.array([np.inf, 100, 10, 5, 2, 1, 0.5, 0.1]) # values for the C parameter in LogisticRegression()
                                                          # C is the INVERSE of regularization strength

In [ ]:

c_values = np.array([np.inf, 100, 10, 5, 2, 1, 0.5, 0.1]) # values for the C parameter in LogisticRegression()
                                                          # C is the INVERSE of regularization strength

def reg_trajectory(tissue: str, c_values: np.array(list[int]), n_splits: int = 10, top_n: int = 20, motif_annotations_path: str = "data/motif_names.tsv") -> pd.DataFrame :

    # Load windows
    X, y, composite = compose_windows(tissue)

    results = []

    for c in c_values:

        # -----------------------
        # Cross-validation
        # -----------------------
        skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=0)
        probs, trues, accs = [], [], []

        base_classifier = LogisticRegression(
            penalty="l2",
            C = c,
            n_jobs=-1
        )

        for i, (train_idx, test_idx) in enumerate(skf.split(X, composite)):
            X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
            y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

            classifier = clone(base_classifier)
            classifier.fit(X_train, y_train)

            p = classifier.predict_proba(X_test)[:, 1]
            probs.append(p)
            trues.append(y_test.values)
            accs.append(accuracy_score(y_test, (p > 0.5).astype(int)))

        # Accuracy metrics
        mean_acc = np.mean(accs)
        std_acc = np.std(accs)

        # ROC AUC metrics
        roc_aucs = []
        for true, prob in zip(trues, probs):
            fpr, tpr, _ = roc_curve(true, prob)
            roc_aucs.append(auc(fpr, tpr))

        mean_auc = np.mean(roc_aucs)
        std_auc = np.std(roc_aucs)

        # Load annotations, then train on full dataset

        annot = pd.read_csv(motif_annotations_path, sep="\t")

        # Build mapping: motif code -> motif name
        code_to_name = (
            annot
            .drop_duplicates("id")
            .set_index("id")["name"]
            .astype(str)
            .to_dict()
        )

        # Map columns; keep original code if annotation is missing
        mapped_columns = (
            X.columns
            .to_series()
            .map(code_to_name)
            .fillna(pd.Series(X.columns, index=X.columns))
        )
        X = X.copy()
        X.columns = mapped_columns

        # train on full dataset
        full_classifier = clone(base_classifier)
        full_classifier.fit(X, y)

        weights = full_classifier.coef_.ravel()

        coef_df = pd.DataFrame({
            "feature": X.columns,
            "weight": weights
        })

        coef_df["abs_weight"] = coef_df["weight"].abs()
        coef_df = coef_df.sort_values("abs_weight", ascending=False)

        top_df = coef_df.head(top_n)

        top_motifs = top_df["feature"].tolist()
        mean_abs_top_weights = top_df["abs_weight"].mean()

        # Store results
        row = {
            "mean abs(top weights)": mean_abs_top_weights,
            "mean AUC ROC": mean_auc,
            "std AUC ROC": std_auc,
            "mean Accuracy": mean_acc,
            "std Accuracy": std_acc
        }

        # Add motif columns: motif 1, motif 2, ...
        for i, motif in enumerate(top_motifs, start=1):
            row[f"motif {i}"] = motif

        results.append(row)

    # Construct final dataframe
    trajectory = pd.DataFrame(results, index=c_values)
    trajectory.index.name = "c parameter value"
    dir = "results/data/regularization_trajectories/sklearn_prototype"
    os.makedirs(dir, exist_ok=True)
    path = os.path.join(dir, f"{tissue}_top_motifs.tsv")
    trajectory.to_csv(path, sep="\t")

    return trajectory

In [ ]:
for t in tissues:
    _ = reg_trajectory(c_values=c_values, tissue=t)